In [ ]:
!npm install -g localtunnel
!pip install flask basicsr facexlib gfpgan realesrgan opencv-python-headless
# Fix for basicsr torchvision issue
!grep -rl 'functional_tensor' /usr/local/lib/python*/dist-packages/basicsr/ | xargs sed -i 's/functional_tensor/functional/g' || true

import os
with open('logic_ai_upscaler.py', 'w', encoding='utf-8') as f:
    f.write('import os\nimport cv2\nimport numpy as np\nimport json\nimport base64\nimport glob\nimport torch\nfrom PIL import Image\n\n# Generative AI Models\nfrom gfpgan import GFPGANer\nfrom realesrgan import RealESRGANer\nfrom basicsr.archs.rrdbnet_arch import RRDBNet\n\ndef update_progress(tracker, task_id, percent, log_msg):\n    if tracker is not None and task_id:\n        tracker[task_id] = {"percent": percent, "log": log_msg}\n    print(f"[PROGRESS] {percent}%: {log_msg}")\n\n_GLOBAL_ENGINES = {}\n\ndef get_cached_engines(model_name, target_scale, device, half_precision, face_restore):\n    global _GLOBAL_ENGINES\n    \n    # We cache based on model configuration\n    cache_key = f"{model_name}_{target_scale}_{device}_{half_precision}"\n    \n    # Select appropriate model structure and path (Mapping all 8 UI options)\n    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)\n    model_path = \'weights/RealESRGAN_v4_General.pth\'\n    url = \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth\'\n    \n    if "Anime" in model_name:\n        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)\n        model_path = \'weights/Anime_Sharp_Illustrations.pth\'\n        url = \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth\'\n    elif "SwinIR" in model_name:\n        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)\n        model_path = \'weights/SwinIR_Texture_Detail.pth\'\n        url = \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth\'\n    elif "BSRGAN" in model_name:\n        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)\n        model_path = \'weights/BSRGAN_Real_World.pth\'\n        url = \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth\'\n    elif "HAT" in model_name:\n        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)\n        model_path = \'weights/HAT_High_Accuracy.pth\'\n        url = \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth\'\n    elif "NAFNet" in model_name:\n        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)\n        model_path = \'weights/NAFNet_Fast_Restoration.pth\'\n        url = \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth\'\n    elif "TextMaster" in model_name:\n        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)\n        model_path = \'weights/TextMaster_V1.pth\'\n        url = \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth\'\n\n    if not os.path.exists(model_path):\n        print(f"        -> [AI ENGINE] Forcefully downloading model weights for {model_name}...")\n        import urllib.request\n        os.makedirs(os.path.dirname(model_path), exist_ok=True)\n        urllib.request.urlretrieve(url, model_path)\n        print(f"        -> [✓] Download complete: {model_path}")\n\n    upsampler_key = f"upsampler_{cache_key}"\n    if upsampler_key not in _GLOBAL_ENGINES:\n        print(f"        -> [AI ENGINE] Initializing RealESRGANer upsampler on {device}...")\n        _GLOBAL_ENGINES[upsampler_key] = RealESRGANer(\n            scale=4,\n            model_path=model_path,\n            model=model,\n            tile=400,\n            tile_pad=10,\n            pre_pad=0,\n            half=half_precision,\n            device=device\n        )\n    upsampler = _GLOBAL_ENGINES[upsampler_key]\n\n    face_enhancer = None\n    if face_restore:\n        face_key = f"gfpgan_{cache_key}"\n        if face_key not in _GLOBAL_ENGINES:\n            print(f"        -> [AI ENGINE] Initializing Face Engine on {device}...")\n            gfpgan_path = \'weights/GFPGAN_Portrait.pth\'\n            if "CodeFormer" in model_name:\n                gfpgan_path = \'weights/CodeFormer_Face_Focus.pth\'\n            \n            if not os.path.exists(gfpgan_path):\n                print(f"        -> [AI ENGINE] Forcefully downloading face model: {os.path.basename(gfpgan_path)}...")\n                import urllib.request\n                gfpgan_url = \'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth\'\n                os.makedirs(os.path.dirname(gfpgan_path), exist_ok=True)\n                urllib.request.urlretrieve(gfpgan_url, gfpgan_path)\n                print(f"        -> [✓] Face Model Download complete: {gfpgan_path}")\n                \n            _GLOBAL_ENGINES[face_key] = GFPGANer(\n                model_path=gfpgan_path,\n                upscale=target_scale,\n                arch=\'clean\',\n                channel_multiplier=2,\n                bg_upsampler=upsampler,\n                device=device\n            )\n        face_enhancer = _GLOBAL_ENGINES[face_key]\n        \n    return upsampler, face_enhancer\n\nclass NexusGenerativeEngine:\n    def __init__(self, payload):\n        print("\\n" + "★"*75)\n        print("⚙️ NEXUS PRO: GENERATIVE ENGINE ⚙️")\n        print("★"*75)\n        \n        self.model_name = payload.get(\'model\', \'RealESRGAN v4\')\n        self.proc_mode = payload.get(\'mode\', \'CPU Precision (Slow)\')\n        print(f"[*] [UI ROUTER] Selected AI Model: {self.model_name}")\n        print(f"[*] [UI ROUTER] Hardware Mode: {self.proc_mode}")\n        \n        # GPU detection based on UI settings\n        self.device = \'cuda\' if (\'GPU\' in self.proc_mode or \'TensorRT\' in self.proc_mode) and torch.cuda.is_available() else \'cpu\'\n        self.half_precision = True if self.device == \'cuda\' else False\n        print(f"[*] [AI ENGINE] Hardware Device: {self.device} (FP16: {self.half_precision})")\n        \n        # Dynamic scale from UI\n        self.target_scale = int(\'\'.join(filter(str.isdigit, str(payload.get(\'factor\', \'4\')).split(\' \')[0])) or 4)\n        \n        adv = json.loads(payload.get(\'settings\', \'{}\'))\n        self.artifact_rem = int(adv.get(\'artifact\', 25))\n        \n        face = json.loads(payload.get(\'face\', \'{}\'))\n        self.face_restore = str(face.get(\'enabled\', \'false\')).lower() == \'true\'\n        \n        # Auto-enable face restore if a portrait model is selected\n        if any(keyword in self.model_name for keyword in ["Face", "Portrait", "CodeFormer", "GFPGAN"]):\n            self.face_restore = True\n            print("[*] [UI ROUTER] Auto-enabled Face Recovery for selected portrait model.")\n            \n        self.face_weight = int(face.get(\'identity\', 85)) / 100.0 \n        \n        export = json.loads(payload.get(\'export\', \'{}\'))\n        self.dpi = int(\'\'.join(filter(str.isdigit, str(export.get(\'dpi\', \'600\')).split(\' \')[0])) or 600)\n        self.color_space = export.get(\'color_space\', \'sRGB\')\n        \n        fmt_str = export.get(\'format\', \'PNG\').upper()\n        if any(x in fmt_str for x in [\'CDR\', \'AI\', \'EPS\']):\n            self.ext, self.pil_fmt = \'eps\', \'EPS\'\n        elif \'PDF\' in fmt_str:\n            self.ext, self.pil_fmt = \'pdf\', \'PDF\'\n        elif any(x in fmt_str for x in [\'PSD\', \'TIFF\', \'TIF\']):\n            self.ext, self.pil_fmt = \'tiff\', \'TIFF\'\n        elif \'WEBP\' in fmt_str:\n            self.ext, self.pil_fmt = \'webp\', \'WEBP\'\n        elif \'JPG\' in fmt_str or \'JPEG\' in fmt_str:\n            self.ext, self.pil_fmt = \'jpg\', \'JPEG\'\n        else:\n            self.ext, self.pil_fmt = \'png\', \'PNG\'\n\n    def process_image(self, input_path, output_dir, tracker=None, task_id=None):\n        update_progress(tracker, task_id, 10, f"Preparing Input File -> {os.path.basename(input_path)}...")\n        img = cv2.imread(input_path)\n        if img is None: \n            raise ValueError("Image corrupted or missing.")\n\n        if not os.path.exists(output_dir): \n            os.makedirs(output_dir)\n\n        update_progress(tracker, task_id, 25, f"Allocating {self.device.upper()} Threads and Loading Architecture...")\n        \n        # Retrieve cached engines\n        upsampler, face_enhancer = get_cached_engines(\n            self.model_name, self.target_scale, self.device, self.half_precision, self.face_restore\n        )\n\n        update_progress(tracker, task_id, 50, f"Processing {self.target_scale}X Upscaling & Generative Enhancements...")\n        if self.face_restore and face_enhancer is not None:\n            if "CodeFormer" in self.model_name:\n                self.face_weight = 0.95\n                \n            _, _, upscaled = face_enhancer.enhance(\n                img, has_aligned=False, only_center_face=False, paste_back=True, weight=self.face_weight\n            )\n        else:\n            upscaled, _ = upsampler.enhance(img, outscale=self.target_scale)\n\n        h, w = upscaled.shape[:2]\n\n        # Gentle Artifact Removal (Optional)\n        if self.artifact_rem > 0:\n            rem_val = max(1, self.artifact_rem // 3)\n            upscaled = cv2.bilateralFilter(upscaled, 5, rem_val, rem_val)\n\n        update_progress(tracker, task_id, 95, f"Formatting for Export ({self.pil_fmt} | {self.dpi} DPI)...")\n        img_rgb = cv2.cvtColor(upscaled, cv2.COLOR_BGR2RGB)\n        pil_master = Image.fromarray(img_rgb)\n        \n        if "CMYK" in self.color_space.upper():\n            pil_master = pil_master.convert(\'CMYK\')\n\n        import time\n        orig_name = os.path.splitext(os.path.basename(input_path))[0]\n        # Use original filename with prefix and unique timestamp to prevent browser cache problems\n        master_file = f"{orig_name}_Nexus_X{self.target_scale}_{int(time.time())}.{self.ext}"\n        master_path = os.path.join(output_dir, master_file)\n        \n        if self.pil_fmt == \'JPEG\':\n            pil_master.save(master_path, format=self.pil_fmt, quality=100, dpi=(self.dpi, self.dpi))\n        elif self.pil_fmt in [\'PDF\', \'EPS\']:\n            pil_master.save(master_path, format=self.pil_fmt, resolution=self.dpi)\n        else:\n            pil_master.save(master_path, format=self.pil_fmt, dpi=(self.dpi, self.dpi))\n\n        _, buffer = cv2.imencode(\'.jpg\', upscaled, [int(cv2.IMWRITE_JPEG_QUALITY), 85])\n        b64_preview = "data:image/jpeg;base64," + base64.b64encode(buffer).decode(\'utf-8\')\n        \n        update_progress(tracker, task_id, 100, "Masterpiece Created Successfully!")\n        \n        # Ensure URLs have leading slash\n        output_url = "/" + master_path.replace("\\\\", "/").lstrip("/")\n        \n        return {\n            "status": "success",\n            "processed_path": output_url, \n            "output_path": output_url,\n            "master_file": output_url, \n            "resolution": f"{w}x{h}",\n            "filename": master_file\n        }\n\ndef process_upscale_logic(data, tracker=None):\n    try:\n        task_id = data.get(\'task_id\', \'default_task\')\n        engine = NexusGenerativeEngine(data)\n        input_file = data.get(\'input_image_path\')\n        if not input_file or not os.path.exists(input_file):\n            return {"status": "error", "message": "Source image missing."}\n        return engine.process_image(input_file, \'static/outputs/\', tracker, task_id)\n    except Exception as e:\n        import traceback\n        print(f"\\n[CRITICAL ERROR] {e}\\n{traceback.format_exc()}")\n        return {"status": "error", "message": str(e)}')
    
import os, time, subprocess
from flask import Flask, request, jsonify, send_file
from logic_ai_upscaler import process_upscale_logic, NexusGenerativeEngine

app = Flask(__name__)
progress_tracker = dict()

@app.route('/api/process-upscale', methods=['POST'])
def upscale():
    file = request.files['image']
    os.makedirs('temp', exist_ok=True)
    input_path = os.path.join('temp', file.filename)
    file.save(input_path)
    
    payload = request.form.to_dict()
    payload['input_image_path'] = input_path
    
    task_id = payload.get('task_id', 'colab_task')
    progress_tracker[task_id] = {"percent": 5, "log": "Starting Colab Engine..."}
    
    # Process
    engine = NexusGenerativeEngine(payload)
    engine.process_image(input_path, 'outputs/', progress_tracker, task_id)
    
    import glob
    files = glob.glob('outputs/*.*')
    if files:
        newest = max(files, key=os.path.getctime)
        return send_file(newest, mimetype='image/jpeg')
    return jsonify({"error": "Upscale failed"})

if __name__ == '__main__':
    # Start localtunnel
    p = subprocess.Popen(["lt", "--port", "5000"], stdout=subprocess.PIPE)
    time.sleep(3)
    out = p.stdout.readline().decode('utf-8').strip()
    url = out.split(" ")[-1]
    print(f'\n\n🔥 COLAB GPU API READY 🔥')
    print(f'COPY THIS URL INTO YOUR LOCAL APP: {url}')
    print(f'='*50 + '\n\n')
    app.run(port=5000)
